LSTM SU IMDb: SENTIMENT ANALYSIS CON KERAS

L'obbiettivo è costruire una rete neurale che riceve una recensione es. cinematografica e restituisce una probabiltà:
- 0 = recensione negativa
- 1 = recensione positiva
Il modello interpreta il risultato come una probabilità, è un problema di classificazione binaria diuna sequenza testuale.

- Padding: come rendere digeribile per la rete la variabilità della lunghezza di un testo
- Layer di Embedding: tecnica per convertire indici interi in vettori densi che catturano relazioni semantiche
- Classificazione binaria del sentiment: l'architettura completa per predire se una recensione è positiva o negativa.

1. DATASET IMDb

Il dataset IMDb è un dataset distribuito di Keras che contiene:
- 25.000 recensioni per il training
- 25.000 recensioni per il test
- etichette positive/negative

Nella versione keras.dataset.imdb il testo è già stato trasformato in sequenze di numeri interi. Ogni numero rappresenta una parola del vocabolario.
Una recensione non appare quindi così:
'this movie was very good'
ma così:
[1,14,22,16,42,530,973]
Quindi i numeri non esprimono ancora il significato delle parole, sono semplicemente identificativi.
14= thes; 22=movie; 16=was; 43=good
La funzione load_data() può limitare il vocabolario alla parole più frequenti attraverso num_words

2. CARICAMENTO DEI DATI

import keras
from keras import layer
VOCAB_SIZE=10_000
(x_traing, y_train),(x_test,y_test)=keras.datasets.imdb.load_data(num_words=VOCAB_SIZE)
Dicendo di considerare solo le 10_000 (VOCAB_SIZE) parolae più frequenti presenti nelle recensioni.

Le parole più rare vengono sostituite con un simbolo che rappresente una parola sconosciuta.

3. IL PROBLEMA DELLA LUNGHEZZA (PADDING)

Le rencensioni non hanno tutte la stessa lunghezza.
Una rete neurale lavora normalmente con batch rappresentati da matrici regolari. Non possiamo crare facilmente una matrice in cui ogni riga ha una lunghezza diversa.
Per questo motivo applichiamo il PADDING.
MAX_LENGTH = 250

x_train = keras.utils.pad_sequences(x_train,maxlen=MAX_LENGTH,padding="post",truncating="post")
x_test = keras.utils.pad_sequences(x_test,maxlen=MAX_LENGTH,padding="post",truncating="post")

Una recensione corta vede riempirsi di zero i caratteri mancanti (zeri posti alla fine o all'inizio, preferito all'inizio)
Una recensione lunga viene invece tagliata (decidendo se all'inizio o alla fine)

La scelta della lunghezza massimo è un equilibirio delicato, se troppo corta perdo informazioni, se troppo lunga aumenta il carico computazionale e la memoria.

L'output finale sarà un vettore di lunghezza fissa pronto per essere convertito in una rappresentazione densa.

4. PERCHE' SERVE L'EMBEDDING

I numeri (esempio 43=good e 18=bad) non possono essere passati così alla rete, perchè non esiste una relazione tra good e bad, pertanto devono essere traformati in vettori densi di n numeri

layers.Embedding(input_dim=10_000,output_dim=128,mask_zero=True)

Questo layer traforma ogni token in un vettore denso di 128 numeri.
prima good=43
dopo good=[0.18,-0.42,0.07,...]

Il vettore viene appreso durante il training.
Parole utilizzate in contesti simili, tenderanno ad avere rappresentazioni simili. Non perchè abbiamo spiegato alla rete il significato di 'is good', ma perchè durante l'addestramento la rete osserva che certe parole contribuiscono frequentemente alla recensioni positive.

Ogni parola quindi, diventa un piccolo array di numeri reali, si utilizzano vettori densi, che a differenza del one-hot encoding, ogni parola è rappresentata da un piccolo vettore di numeri reali (es. 128 dimensioni). 

5. COSA FA LA LSTM

La LSTM legge una recensione una parola alla volta
Consideriamo:
' The movie was not good'
La parola 'good', isolatamente, suggerisce un sentiment positivo.
Ma la sequenza
'not good'
La rete deve ricordare il 'not' quando arriva a 'good'
Una rete Dense tradizionale non gestisce naturalmente l'ordine temporale delle parole. La LSTM mantiene invece uno stato iterno, aggiornato a ogni token.
The -> aggiorna memoria
movie -> aggiorna memoria
was -> aggiorna memoria
not -> conserva informazione negativa
good -> interpreta good tenendo conto di not

La LSTM è progettata per controllare quali informaioni: - memorizzare - dimenticare - utilizzare per produrre l'output
Keras fornisce il layer LSTM, che può usare implementazioni ottimizzate su GPU quando configurazione e backend lo permettono

6. IL PRIMO MODELLO COMPLETO

import keras
from keras import layers

VOCAB_SIZE = 10_000
MAX_LENGTH = 250
EMBEDDING_DIM = 128
LSTM_UNITS = 64

model = keras.Sequential([
    layers.Input(shape=(MAX_LENGTH,)),
    layers.Embedding(input_dim=VOCAB_SIZE,output_dim=EMBEDDING_DIM,mask_zero=True),
    layers.LSTM(units=LSTM_UNITS),
    layers.Dense(units=1,activation="sigmoid")
])
model.summary()

vediamo ogni layer

7. INPUT LAYER
layers.Input(shape=(MAX_LENGTH,))

Ogni esempio è una sequenza di 250 numeri.
Se un batch di 32 esempi, la forma sarà: (32,250)
Il batch non viene scritto nell'input, perchè può variare

8. EMBEDDING LAYER
layers.Embedding(input_dim=VOCAB_SIZE,output_dim=EMBEDDING_DIM,mask_zero=True)

input_dim: dimensione del vocabolario (10_000 nell'esempio)
output_dim: ogni parola viene rappresentata da n valori (128 nell'esempio)
La matrice dei pesi dell'Embedding avrà la forma (10_000,18)
Numero di parametri 10_000 x 128 = 1_280_000
Con mask_zero=True, il modello cerca di ignorare le posizioni aggiunge artificialmente, e queste posizioni aggiunte non devono influenzare il sentiment.

9. LSTM LAYER
layers.LSTM(units=LSTM_UNITS)

unit=64 significa che lo stato interno della LSTM contiene 64 valori
La LSTM restituisce soltanto il risultato finale (batch_size,64)
comprime l'intera recensione in un vettore di 64 valori

Esempio 
recensione di 250 parole
LSTM
[0.43,-0.18,0.92,....,0.11]
Questo vettore rappresenta la recensione
Ma non è il 'significato completo' della recensione. sono le informazioni che la rete ha imparato a considerare utili per la classificazione.

10. OUTPUT DENSE
layers.Dense(units=1,activation="sigmoid")

Il layer riceve 64 valori prodotti dalla LSTM e restituisce un solo numero
La sigmoid produce valori tra 0 e 1
0.07 -> probabilmente negativa
0.48 -> molto incerta
0.91 -> probabilmente positiva
La soglia standard è 0.5
classe = 1 if probabilità >= 0.5 else 0
La soglia standard (0.5) non è legge, in un progetto reale può essere modificata in funzione dei costi degli errori.

11. COMPILAZIONE
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

optimazer='adam': Adam aggiorna i pesi della rete cercando di ridurre la loss
loss='binary_crossentropy': adatta ad una classificazione binaria con output sigmoid
La loss penalizza il modello quando la probabilità prevista è lontana dall'etichetta reale.
accuracy= metrics['accuracy']: misura la percentuale di classificazioni corrette.

12. ADDESTRAMENTO
history = model.fit(
    x_train,
    y_train,
    validation_split=0.2,
    epochs=5,
    batch_size=64
)

validation_splot=0.2
Il 20% del training set viene usato per la validazione
opochs=5 il modello legge l'intero training set per 5 volte
batch_size=64: i pesi vengono aggiornati dopo ogni gruppo di 64 recensioni

13. VALUTAZIONE DEL TEST SET
test_loss, test_accuracy = model.evaluate(
    x_test,
    y_test,
    verbose=0
)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

- training_set: usato per imparare
- validation set: usato per controllare il comportamente durante lo sviluppo
- test set: usato soltanto alla fine per stimare le prestazioni su dati mai visti

14. PREVISIONE SU UNA RECENSIONE DEL DATASET
probabilita = model.predict(x_test[:1], verbose=0)[0][0]

classe = 1 if probabilita >= 0.5 else 0

print("Probabilità positiva:", probabilita)
print("Classe prevista:", classe)
print("Classe reale:", y_test[0])

La LSTM standard legge il testo in una sola direzione.
Quando analizza una parola, conosce ciò che è apparso prima, ma non ciò che apparirà dopo.
Una BIDIRECTIONALE LSTM legge la sequenza in entrambe le direzioni
prima -> ultima
ultima -> prima




In [1]:
import keras
from keras import layers, models
from keras.datasets import imdb
from keras.preprocessing.sequence import pad_sequences

# --- 1. CONFIGURAZIONE E CARICAMENTO DATI ---
# Limiteremo il vocabolario alle 10.000 parole più frequenti.
# Teoria: Ridurre il vocabolario aiuta a gestire la dimensionalità e rimuove il "rumore" di parole rare.
max_features = 10000 
# Ogni recensione verrà tagliata o allungata a 200 parole.
# Teoria: Il padding è essenziale perché i tensori in un batch devono avere dimensioni uniformi.
maxlen = 200 

print("Caricamento dati...")
# caricamento asincrono integrato di Keras
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

# --- 2. PREPROCESSING: PADDING ---
# Usiamo pad_sequences per trasformare le liste di interi in un array NumPy (Batch, Maxlen)
# Teoria: 'pre' padding è spesso preferito nelle LSTM perché il segnale finale (non-zero) 
# è più vicino al calcolo dell'output finale, riducendo la dissipazione dell'informazione.
x_train = pad_sequences(x_train, maxlen=maxlen, padding='pre')
x_test = pad_sequences(x_test, maxlen=maxlen, padding='pre')

# --- 3. DEFINIZIONE DELL'ARCHITETTURA ---
model = models.Sequential([
    # Layer di Embedding: trasforma indici (0-10000) in vettori densi di dimensione 128.
    # Teoria: A differenza del One-Hot, l'Embedding impara coordinate spaziali dove 
    # parole con contesti simili (es. 'bello', 'fantastico') finiscono vicine.
    layers.Embedding(input_dim=max_features, output_dim=128, name="Semantics_Space"),
    
    # Layer LSTM: processa la sequenza di vettori.
    # Teoria: Grazie alle sue 'gate', la LSTM decide quali parole della recensione 
    # sono cruciali per il sentiment (es. 'ma', 'nonostante', 'ottimo') e quali ignorare.
    layers.LSTM(64, dropout=0.2, recurrent_dropout=0.2, name="Context_Processor"),
    
    # Layer Finale: un singolo neurone con attivazione Sigmoide.
    # Teoria: La sigmoide schiaccia l'output in [0, 1]. 0 = Negativo, 1 = Positivo.
    layers.Dense(1, activation='sigmoid', name="Sentiment_Classifier")
])

# --- 4. COMPILAZIONE E TRAINING ---
# Usiamo binary_crossentropy perché abbiamo solo due classi (0 o 1).
model.compile(optimizer='adam', 
              loss='binary_crossentropy', 
              metrics=['accuracy'])

print("Inizio addestramento...")
# Teoria: Usiamo un validation_split per monitorare se il modello sta imparando a 
# generalizzare o se sta solo memorizzando le recensioni (overfitting).
history = model.fit(x_train, y_train, 
                    epochs=3, 
                    batch_size=32, 
                    validation_split=0.2)

# Valutazione finale sui dati mai visti (Test Set)
results = model.evaluate(x_test, y_test)
print(f"Test Loss: {results[0]:.4f}, Test Accuracy: {results[1]:.4f}")






# Recuperiamo il dizionario che associa le parole ai numeri (word index)
word_index = imdb.get_word_index()

def predict_sentiment(text):
    # 1. Preprocessing del testo: pulizia e conversione in minuscolo
    words = text.lower().split()
    
    # 2. Traduzione parole -> indici (considerando l'offset di 3 usato da Keras IMDB)
    # 0 = padding, 1 = start, 2 = unknown.
    tokenized = []
    for word in words:
        index = word_index.get(word, -3) # -3 perché poi aggiungiamo 3
        actual_index = index + 3
        if actual_index < max_features:
            tokenized.append(actual_index)
        else:
            tokenized.append(2) # Segna come parola "Unknown"
            
    # 3. Padding per arrivare a 200 parole
    padded_text = pad_sequences([tokenized], maxlen=maxlen)
    
    # 4. Predizione
    prediction = model.predict(padded_text, verbose=0)[0][0]
    
    sentiment = "POSITIVA" if prediction > 0.5 else "NEGATIVA"
    confidenza = prediction if prediction > 0.5 else 1 - prediction
    
    print(f"\nRecensione: \"{text}\"")
    print(f"Sentiment: {sentiment} ({confidenza*100:.2f}% di confidenza)")

# --- ESEMPI DI TEST ---
print("\n" + "="*30)
print("TEST SU NUOVE RECENSIONI")
print("="*30)

# Esempio 1: Chiaramente positivo
predict_sentiment("this movie was awesome and the acting was fantastic I loved it")

# Esempio 2: Chiaramente negativo
predict_sentiment("this movie was a complete waste of time the plot was terrible")

# Esempio 3: Sfumato/Neutro
predict_sentiment("the beginning was good but the ending was very boring and slow")

Caricamento dati...
17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
Inizio addestramento...
Epoch 1/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 214s 324ms/step - accuracy: 0.7616 - loss: 0.4906 - val_accuracy: 0.8174 - val_loss: 0.4065
Epoch 2/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 140s 224ms/step - accuracy: 0.8596 - loss: 0.3403 - val_accuracy: 0.8274 - val_loss: 0.4610
Epoch 3/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 135s 216ms/step - accuracy: 0.8786 - loss: 0.2999 - val_accuracy: 0.8238 - val_loss: 0.3930
782/782 ━━━━━━━━━━━━━━━━━━━━ 50s 63ms/step - accuracy: 0.8371 - loss: 0.3836
Test Loss: 0.3836, Test Accuracy: 0.8371
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step

TEST SU NUOVE RECENSIONI

Recensione: "this movie was awesome and the acting was fantastic I loved it"
Sentiment: POSITIVA (97.10% di confidenza)

Recensione: "this movie was a complete waste of time the plot was terrible"
Sentiment: NEGATIVA (98.10% di confidenza)

Recensione: "the beginning was good but the ending was very boring and slow"


Con il dataset IMDb è possibile creare da zero un piccolo strumento di sentiment analisys.
